**Użytkownicy Google Colab** — jeśli brakuje `lm_utils.py`, odkomentuj komórkę poniżej.
W repozytorium sympozjum (lokalnie / Docker / Cloud Agent) plik jest w `notebook/`.

In [ ]:
# import os
# if not os.path.exists('lm_utils.py'):
#     !curl -o lm_utils.py https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/notebook/lm_utils.py

In [ ]:
import sys
from pathlib import Path

for p in (Path.cwd(), Path.cwd() / "notebook", Path.cwd().parent / "notebook"):
    if (p / "lm_utils.py").exists():
        sys.path.insert(0, str(p))
        break

from lm_utils import ensure_package

ensure_package("pandas")
ensure_package("numpy")
ensure_package("matplotlib")
ensure_package("tensorflow")

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lm_utils import display_generated_name, reset_seed, visualize_probabilities

# Powtarzalność treningu i generacji na sali (sympozjum: stałe ziarno).
SEED = 42
LICZBA_EPOK = 12
LEARNING_RATE = 0.003
TEMPERATURA = 0.4
EMBEDDING_DIM = 64
GRU_UNITS = 128
DROPOUT = 0.2

# Model Języka (Language Model) — generujemy łacińskie nazwy chorób

Ten sam mechanizm co w ChatGPT, Claude czy Gemini: model szacuje **warunkowy rozkład** kolejnego tokenu.

Przy zdaniu *Mamy dzisiaj bardzo słoneczną …* przewidujemy `P(kolejne słowo | dotychczasowy tekst)`. Tu **token = pojedyncza litera**, a „język” to nazwy chorób z pliku w repozytorium sympozjum.

## Dlaczego poziom liter, nie słów?

Słownik słów miałby setki tysięcy wpisów. Przy ~30 znakach (litery, spacja, `-`, `%`, `!`) możemy wytrenować **mały model na laptopie w ~15 sekund** i pokazać cały pipeline: tokeny → embedding → GRU → softmax → losowanie z temperaturą.

## Wczytanie danych

Zbiór `data/choroby.csv` — łacińskie nazwy chorób (Wikipedia / ICD-style), jedna nazwa na linię. Sympozjum używa **1292 unikalnych nazw** po deduplikacji (ziarno 42 tasuje kolejność).

In [ ]:
REPO = "https://raw.githubusercontent.com/NXTRSS/sympozjum-pediatria/main/data"

zrodlo = f"{REPO}/choroby.csv"
plik = "choroby.csv"
for kandydat in (plik, f"data/{plik}", f"../data/{plik}"):
    if os.path.exists(kandydat):
        zrodlo = kandydat
        break

df = pd.read_csv(zrodlo, header=None, names=["nazwa"], sep="\t")
print(f"Wczytano z: {zrodlo}")
df.head()

Przykładowe nazwy z początku listy (po tasowaniu ziarnem):

In [ ]:
list_of_names = (
    df["nazwa"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
rng = np.random.default_rng(SEED)
list_of_names = [list_of_names[i] for i in rng.permutation(len(list_of_names))]

print(f"Liczba nazw: {len(list_of_names)}")
print("Pierwsze 10:", list_of_names[:10])

## Tokeny początku i końca sekwencji

- `%` — *start*: „zacznij pisać nazwę”
- `!` — *stop*: „koniec nazwy”

`pneumonia` → `%pneumonia!`. Model uczy się, kiedy wypisać `!` zamiast generować w nieskończoność.

Dodaj `%` i `!` do każdej nazwy (list comprehension):

In [ ]:
name_list = ["%" + name + "!" for name in list_of_names]
print(name_list[:5])

## Tokenizacja — zamiana liter na liczby

`Tokenizer(char_level=True)` przypisuje każdej literze (i `%`, `!`) numer. Indeks `0` zostawiamy na **padding** (wyrównanie długości nazw w macierzy).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import tensorflow as tf
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow.keras as keras

In [ ]:
reset_seed(SEED)

tokenizer = Tokenizer(filters="", lower=True, char_level=True)
tokenizer.fit_on_texts(name_list)

vocab_size = len(tokenizer.word_index) + 1
print(f"Wielkość słownika (z paddingiem 0): {vocab_size}")
print("Przykłady znak → indeks:")
for i, (ch, idx) in enumerate(tokenizer.word_index.items()):
    if i >= 8:
        break
    print(f"  {ch!r}: {idx}")

## Czym jest X i Y?

Dla `%pneumonia!` uczymy **przesunięcie o jeden znak**:

```
X = %pneumonia   →   Y = pneumonia!
```

Jedno przejście GRU po sekwencji uczy wszystkie prefiksy naraz: `P(następna litera | kontekst)`.

## Padding

Nazwy mają różne długości. `pad_sequences` dopełnia krótsze zerami z prawej. W `Embedding(mask_zero=True)` zera **nie wchodzą do straty**.

In [ ]:
# X: bez ostatniego znaku, Y: bez pierwszego
X_strings = [name[:-1] for name in name_list]
y_strings = [name[1:] for name in name_list]
print(f"Przykład: X={X_strings[0]!r}  →  Y={y_strings[0]!r}")

X_train = pad_sequences(tokenizer.texts_to_sequences(X_strings), padding="post")
y_train = pad_sequences(tokenizer.texts_to_sequences(y_strings), padding="post")

print("Po tokenizacji — wejście[0]:", X_train[0])
print("Po tokenizacji — wyjście[0]:", y_train[0])

In [ ]:
X_train.shape

Kształt `(nazwy, max_długość)` — każdy wiersz to jedna choroba dopełniona do wspólnej długości. **Y to X przesunięte o 1.**

## Konstrukcja modelu

1. **Embedding(64)** — numer litery → wektor  
2. **GRU(128, dropout=0.2)** — kontekst wzdłuż nazwy; `return_sequences=True`, `return_state=True` (do szybkiej generacji)  
3. **Dense + softmax** — rozkład następnej litery  

Nastawy sympozjum (eksperymenty w repo): 12 epok, Adam `lr=0.003`, temperatura `0.4` przy losowaniu.

In [ ]:
reset_seed(SEED)


class LanguageModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding(
            vocab_size,
            output_dim=EMBEDDING_DIM,
            mask_zero=True,
            embeddings_initializer=keras.initializers.GlorotUniform(seed=SEED),
        )
        self.rnn = GRU(
            GRU_UNITS,
            return_sequences=True,
            return_state=True,
            dropout=DROPOUT,
            seed=SEED,
            kernel_initializer=keras.initializers.GlorotUniform(seed=SEED),
            recurrent_initializer=keras.initializers.Orthogonal(seed=SEED + 1),
            bias_initializer=keras.initializers.Zeros(),
        )
        self.fc = Dense(
            vocab_size,
            activation="softmax",
            kernel_initializer=keras.initializers.GlorotUniform(seed=SEED + 2),
            bias_initializer=keras.initializers.Zeros(),
        )

    def call(self, inputs, training=False, hidden_state=None, return_state=False):
        x = self.embedding(inputs, training=training)
        seq_out, last_hidden_state = self.rnn(
            x, training=training, initial_state=hidden_state
        )
        x = self.fc(seq_out)
        if return_state:
            return x, last_hidden_state
        return x


model = LanguageModel()
model(X_train)
model.summary()

## Trening

Sparse Categorical Crossentropy — etykieta to indeks litery (jak `y_train`). Bez osobnego zbioru walidacyjnego (notatnik pokazowy).

In [ ]:
reset_seed(SEED)

criterion = SparseCategoricalCrossentropy()
model.compile(
    loss=criterion,
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
)

historia = model.fit(
    X_train,
    y_train,
    epochs=LICZBA_EPOK,
    batch_size=64,
    shuffle=True,
    verbose=2,
)

### Krzywa uczenia

In [ ]:
from lm_wizualizacja import wykres_krzywy_bledu

wykres_krzywy_bledu(historia, tytul="Krzywa uczenia — błąd w trakcie epok")

## Generacja — wersja z hidden state

Zamiast przepuszczać cały rosnący prefiks od nowa (wolno), **zapamiętujemy hidden state GRU** i w kolejnych krokach podajemy tylko nową literę — ten sam pomysł co *KV cache* w dużych LLM.

**Temperatura** `0.4`: `log(p)/T` przed losowaniem — niżej = ostrożniej, wyżej = bardziej zróżnicowane nazwy.

In [ ]:
def _wybierz_litere(last_output, temperature, greedy=False):
    p = np.array(last_output, dtype=np.float64)
    if greedy:
        p[0] = 0
        return int(p.argmax())
    p = np.log(p + 1e-9) / temperature
    p = np.exp(p)
    p[0] = 0
    if p.sum() == 0:
        return int(np.argmax(last_output))
    p /= p.sum()
    return int(np.random.choice(len(p), p=p))


def sample_model_optimal(
    prompt="%",
    greedy=False,
    temperature=TEMPERATURA,
    print_outcome=True,
    visuals=False,
    known_names=None,
):
    encoded = tf.convert_to_tensor(tokenizer.texts_to_sequences([prompt]))
    hidden_state = None
    known_names = set(known_names or list_of_names)

    for krok in range(40):
        model_out, hidden_state = model(encoded, return_state=True, hidden_state=hidden_state)
        last_output = model_out[0, -1, :].numpy()
        next_idx = _wybierz_litere(last_output, temperature, greedy=greedy)
        if visuals:
            visualize_probabilities(
                last_output, krok, tokenizer, prompt, chosen_idx=next_idx, show=False
            )
        next_char = tokenizer.index_word[next_idx]
        prompt += next_char
        encoded = tf.convert_to_tensor(tokenizer.texts_to_sequences([next_char]))
        if next_char == "!":
            break

    if print_outcome:
        display_generated_name(prompt, known_names)
    if visuals:
        plt.show()
    return prompt

In [ ]:
reset_seed(SEED)

prompt = "%"
pelny = sample_model_optimal(
    prompt, visuals=True, greedy=False, known_names=set(list_of_names)
)
pierwsza_nazwa = pelny[1:-1]  # bez % i !
print("Wygenerowano:", pierwsza_nazwa)

Na wykresach widać **rozkład P(następna litera | kontekst)** w każdym kroku (pomarańczowy słupek = wylosowany znak). Im dłuższy kontekst, tym często ostrzejszy rozkład — aż dominuje `!`.

## Dziesięć nazw — czy model coś nowego wymyśla?

In [ ]:
reset_seed(SEED)

lista_nazw = []
znane = set(list_of_names)
nowe = 0
for _ in range(10):
    wynik = sample_model_optimal("%", print_outcome=False, visuals=False, known_names=znane)
    nazwa = wynik[1:-1]
    lista_nazw.append(nazwa)
    if nazwa in znane:
        print(f"  • {nazwa:<36} ⚠️  w zbiorze")
    else:
        print(f"  • {nazwa:<36} ✨ nowa")
        nowe += 1
print(f"\n{nowe}/10 nazw nie występuje w zbiorze treningowym.")

## Skala

Ten model ma ~80 tys. parametrów. ChatGPT ma miliardy i Transformer zamiast GRU — ale **rdzeń jest ten sam**: softmax następnego tokenu, autoregresja, temperatura przy losowaniu.

*Aplikację dla sali uruchamia prelegent: `notebook/uruchom_demo.py`.*